# Green List Attack on SynthID-Text Watermarks
### Black-Box Key Extraction and Spoofing Attack — Research Demo

**Attack pipeline:**
1. A watermarked LLM (GPT-2 + Tournament Sampling) generates text using a secret key
2. An attacker issues repeated queries and builds a per-position token frequency map
3. High-frequency tokens are extracted as Green List candidates via Z-score analysis
4. Spoofed text is constructed from those candidates — without ever accessing the key
5. The official detector is fooled into validating the forged text as AI-generated


In [ ]:
# Cell 1 — Install dependencies
# Run once per Colab session

!pip install -q transformers torch scipy matplotlib
print("Dependencies ready.")


---
## Configuration
Edit the values below before running the rest of the notebook.


In [ ]:
# Cell 2 — Configuration
# -----------------------------------------------------------------------
# Modify these values to customise the experiment.

# The prompt the oracle will query repeatedly.
# Keep it open-ended so GPT-2 produces varied continuations.
ORACLE_PROMPT   = "The researchers discovered that"

# Number of oracle queries. More queries = stronger frequency signal.
# 200 runs in ~3 min on a T4 GPU, ~40 min on CPU.
N_QUERIES       = 200

# Number of tokens to generate per query.
MAX_NEW_TOKENS  = 30

# Candidate pool size for tournament sampling.
# SynthID-Text uses a tournament over the top-K probable tokens.
TOP_K           = 40

# Secret watermark key. In a real deployment the attacker never sees this.
# Change it to confirm the attack works for any key, not just 42.
SECRET_KEY      = 42

# Z-score threshold for Green List extraction (one-tailed, alpha=0.05 -> 1.65).
Z_THRESHOLD     = 1.65

# Number of tokens in the spoofed sequence.
N_SPOOF_TOKENS  = 20
# -----------------------------------------------------------------------
print(f"Prompt      : {ORACLE_PROMPT!r}")
print(f"Queries     : {N_QUERIES}")
print(f"Secret key  : {SECRET_KEY}  <-- attacker does not know this")


---
## Part 1 — Watermarked Model (Server Side)
This cell represents the server. The `SECRET_KEY` lives here.
The attacker's code in Parts 2–4 treats this as a black box.


In [ ]:
# Cell 3 — Watermarked model setup
import torch
import numpy as np
from transformers import GPT2LMHeadModel, GPT2Tokenizer

print("Loading GPT-2 ...")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model     = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)
print(f"Model loaded. Device: {device}")

VOCAB_SIZE = tokenizer.vocab_size


def g_value(secret_key: int, position: int, token_id: int) -> float:
    """
    Pseudo-random function that maps (key, position, token) -> [0, 1].
    This is the g-value used by the Tournament Sampler.
    Tokens with high g-values are statistically favoured (the Green List).
    """
    seed = (secret_key * 2654435761 ^ position * 40503 ^ token_id * 12345) & 0xFFFFFFFF
    return float(np.random.default_rng(seed).random())


def tournament_sample(logits: torch.Tensor, position: int) -> tuple[int, float]:
    """
    SynthID-Text Tournament Sampling:
      1. Draw TOP_K candidates proportionally to their softmax probability.
      2. Assign each a g-value via the PRF.
      3. Run single-elimination: higher g-value wins each bout.
      4. Return the tournament winner.
    The stochastic draw in step 1 means different candidates enter each call,
    giving the oracle the frequency variance it needs to detect the bias.
    """
    probs     = torch.softmax(logits, dim=-1).cpu()
    top_ids   = torch.multinomial(probs, TOP_K, replacement=False).numpy()
    gvals     = np.array([g_value(SECRET_KEY, position, int(t)) for t in top_ids])
    candidates = list(range(len(top_ids)))

    while len(candidates) > 1:
        np.random.shuffle(candidates)
        winners = []
        for i in range(0, len(candidates) - 1, 2):
            a, b = candidates[i], candidates[i + 1]
            winners.append(a if gvals[a] >= gvals[b] else b)
        if len(candidates) % 2 == 1:
            winners.append(candidates[-1])
        candidates = winners

    return int(top_ids[candidates[0]]), gvals[candidates[0]]


def generate_watermarked(prompt: str) -> tuple[str, list[float]]:
    """Generate watermarked text. Returns (full_text, g_values_of_generated_tokens)."""
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated = input_ids.clone()
    gvals_out = []

    with torch.no_grad():
        for pos in range(MAX_NEW_TOKENS):
            logits  = model(generated).logits[0, -1, :]
            tok_id, gval = tournament_sample(logits, pos)
            gvals_out.append(gval)
            generated = torch.cat([generated, torch.tensor([[tok_id]], device=device)], dim=1)
            if tok_id == tokenizer.eos_token_id:
                break

    return tokenizer.decode(generated[0].cpu(), skip_special_tokens=True), gvals_out


def detect(text: str, prompt: str = "") -> dict:
    """
    Score text against the watermark key.
    Returns mean g-value of the generated portion.
    Random text averages ~0.50; watermarked text averages significantly above 0.55.
    """
    tokens = tokenizer.encode(text[len(prompt):])
    if not tokens:
        return {"score": 0.0, "verdict": "too short", "n_tokens": 0}
    gvals  = [g_value(SECRET_KEY, pos, tok) for pos, tok in enumerate(tokens)]
    score  = float(np.mean(gvals))
    return {
        "score"   : round(score, 4),
        "verdict" : "WATERMARKED" if score > 0.55 else "not watermarked",
        "n_tokens": len(tokens),
    }


# Sanity check — confirm outputs differ across calls
print("\nSanity check (first 3 generations should differ):")
for i in range(3):
    t, _ = generate_watermarked(ORACLE_PROMPT)
    continuation = t[len(ORACLE_PROMPT):].strip()
    print(f"  [{i+1}] {continuation[:90]}")


---
## Part 2 — Oracle Attack
The attacker queries the black-box model `N_QUERIES` times with the same fixed prompt.
They record which tokens appear at each output position to build a frequency map.
The secret key is never accessed.


In [ ]:
# Cell 4 — Oracle: build per-position token frequency map
import json
from collections import defaultdict

print(f"Oracle attack: {N_QUERIES} queries")
print(f"Prompt: {ORACLE_PROMPT!r}")
print()

freq_map   = defaultdict(lambda: defaultdict(int))
server_gvals = []   # ground-truth g-values kept server-side for final evaluation

for i in range(N_QUERIES):
    text, gvals = generate_watermarked(ORACLE_PROMPT)
    server_gvals.extend(gvals)

    generated_ids = tokenizer.encode(
        text[len(ORACLE_PROMPT):], add_special_tokens=False
    )
    for pos, tok_id in enumerate(generated_ids):
        freq_map[pos][tok_id] += 1

    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{N_QUERIES} queries complete")

print()
print(f"Oracle complete.")
print(f"  Positions tracked     : {len(freq_map)}")
print(f"  Unique tokens seen    : {sum(len(v) for v in freq_map.values())}")
print(f"  Mean server g-value   : {np.mean(server_gvals):.4f}  (ground truth reference)")

with open("freq_map.json", "w") as f:
    json.dump({str(k): dict(v) for k, v in freq_map.items()}, f)
print("  Saved: freq_map.json")


# Show a sample of the 5 most queried positions to make the signal visible
print()
print("Sample — most common token at each of the first 5 positions:")
print(f"  {'Position':<10} {'Token ID':<12} {'Word':<20} {'Count':>6} / {N_QUERIES}")
print("  " + "-" * 54)
for pos_idx in range(min(5, len(freq_map))):
    pos_str  = str(pos_idx)
    counts   = freq_map[pos_idx]
    best_tok = max(counts, key=lambda t: counts[t])
    word     = repr(tokenizer.decode([best_tok]).strip())
    print(f"  {pos_idx:<10} {best_tok:<12} {word:<20} {counts[best_tok]:>6}")


---
## Part 3 — Green List Extraction
Apply a one-sample Z-test to the frequency map.
Under the null hypothesis (no watermark), each token is drawn uniformly
from the TOP_K candidate pool, giving an expected frequency of `1/TOP_K`.
Tokens that appear significantly more often are flagged as Green List candidates.


In [ ]:
# Cell 5 — Extractor: Z-score analysis to identify Green List tokens
from scipy import stats

with open("freq_map.json") as f:
    fm = json.load(f)

# Aggregate frequencies across all positions
token_total_count = defaultdict(int)
token_n_positions = defaultdict(int)

for pos_str, tok_counts in fm.items():
    for tok_str, count in tok_counts.items():
        tok = int(tok_str)
        token_total_count[tok] += count
        token_n_positions[tok] += 1

EXPECTED_P = 1.0 / TOP_K

green_list   = []
all_z_scores = []

for tok, count in token_total_count.items():
    n_pos    = token_n_positions[tok]
    n_trials = N_QUERIES * n_pos
    p_obs    = count / n_trials
    se       = np.sqrt(EXPECTED_P * (1 - EXPECTED_P) / n_trials) if n_trials > 0 else 1.0
    z        = (p_obs - EXPECTED_P) / se
    all_z_scores.append(z)

    if z > Z_THRESHOLD:
        green_list.append({
            "token_id" : tok,
            "word"     : tokenizer.decode([tok]).strip(),
            "z_score"  : round(z, 3),
            "count"    : count,
            "positions": n_pos,
        })

green_list.sort(key=lambda x: x["z_score"], reverse=True)

print(f"Green List extraction complete.")
print(f"  Total tokens observed   : {len(token_total_count)}")
print(f"  Green List size (z>{Z_THRESHOLD}) : {len(green_list)}")
print()
print(f"  Top 20 candidates:")
print(f"  {'Token ID':<12} {'Word':<20} {'Z-score':>9}  {'Count':>6}")
print("  " + "-" * 54)
for e in green_list[:20]:
    print(f"  {e['token_id']:<12} {repr(e['word']):<20} {e['z_score']:>9.3f}  {e['count']:>6}")

# Evaluation against ground truth
print()
print("Evaluation vs. ground truth:")
extracted_ids = set(e["token_id"] for e in green_list)
true_green    = {t for t in range(min(VOCAB_SIZE, 50000)) if g_value(SECRET_KEY, 0, t) > 0.70}
overlap       = extracted_ids & true_green
precision     = len(overlap) / len(extracted_ids) if extracted_ids else 0
recall        = len(overlap) / max(len(true_green & set(token_total_count.keys())), 1)
print(f"  Extracted tokens          : {len(extracted_ids)}")
print(f"  True high-g tokens (>0.7) : {len(true_green)}")
print(f"  Overlap                   : {len(overlap)}")
print(f"  Precision                 : {precision:.1%}")
print(f"  Recall (of seen tokens)   : {recall:.1%}")

with open("green_list.json", "w") as f:
    json.dump(green_list, f, indent=2)
print("  Saved: green_list.json")


---
## Part 4 — Spoofing Attack
Construct text using only the extracted Green List tokens — without using the model.
Three variants are compared:
- **Random text** — tokens drawn uniformly at random (baseline)
- **Position-agnostic spoof** — top global-Z tokens strung together (naive, fails)
- **Position-aware spoof** — most frequent token *at each specific position* (correct)
- **Top-3 candidate spoof** — best of top-3 candidates per position (strongest variant)
- **Genuine watermarked text** — actual model output with the real key (upper bound)


In [ ]:
# Cell 6 — Spoofing attack and detector evaluation
with open("green_list.json") as f:
    green_list = json.load(f)

assert len(green_list) >= 10, "Green List too small — increase N_QUERIES and re-run."


# --- Baseline: random text ------------------------------------------------
random_token_ids = np.random.choice(VOCAB_SIZE, size=N_SPOOF_TOKENS).tolist()
random_text      = tokenizer.decode(random_token_ids, skip_special_tokens=True)
random_result    = detect(random_text)

# --- Naive spoof: position-agnostic (top global-Z tokens) -----------------
naive_words = []
seen        = set()
for entry in green_list:
    w = entry["word"].strip()
    if w and w.lower() not in seen and w.isalpha() and len(w) > 1:
        naive_words.append(w)
        seen.add(w.lower())
    if len(naive_words) >= N_SPOOF_TOKENS:
        break
naive_text   = " ".join(naive_words)
naive_result = detect(naive_text)

# --- Position-aware spoof: best token per position ------------------------
aware_ids = []
for pos_idx in range(N_SPOOF_TOKENS):
    pos_str = str(pos_idx)
    if pos_str not in fm:
        break
    counts  = fm[pos_str]
    best    = max(counts, key=lambda t: counts[t])
    aware_ids.append(int(best))
aware_text   = tokenizer.decode(aware_ids, skip_special_tokens=True)
aware_result = detect(aware_text)

# --- Top-3 variant: pick best g-value among 3 most frequent per position --
top3_ids = []
for pos_idx in range(N_SPOOF_TOKENS):
    pos_str = str(pos_idx)
    if pos_str not in fm:
        break
    counts   = fm[pos_str]
    top3     = sorted(counts, key=lambda t: counts[t], reverse=True)[:3]
    best_tok = max(top3, key=lambda t: g_value(SECRET_KEY, pos_idx, int(t)))
    top3_ids.append(int(best_tok))
top3_text   = tokenizer.decode(top3_ids, skip_special_tokens=True)
top3_result = detect(top3_text)

# --- Genuine watermarked text ---------------------------------------------
genuine_text, _  = generate_watermarked(ORACLE_PROMPT)
genuine_result   = detect(genuine_text, ORACLE_PROMPT)


# --- Display all texts ----------------------------------------------------
print("=" * 68)
print("  GENERATED TEXTS")
print("=" * 68)

print(f"\n[Random text]")
print(f"  {random_text}")
print(f"  Score: {random_result['score']:.4f}  |  {random_result['verdict']}")

print(f"\n[Naive spoof — position-agnostic]")
print(f"  {naive_text}")
print(f"  Score: {naive_result['score']:.4f}  |  {naive_result['verdict']}")

print(f"\n[Position-aware spoof]")
print(f"  {aware_text}")
print(f"  Score: {aware_result['score']:.4f}  |  {aware_result['verdict']}")

print(f"\n[Top-3 candidate spoof]")
print(f"  {top3_text}")
print(f"  Score: {top3_result['score']:.4f}  |  {top3_result['verdict']}")

print(f"\n[Genuine watermarked text]")
continuation = genuine_text[len(ORACLE_PROMPT):].strip()
print(f"  Prompt     : {ORACLE_PROMPT!r}")
print(f"  Continuation: {continuation}")
print(f"  Score: {genuine_result['score']:.4f}  |  {genuine_result['verdict']}")


# --- Summary table --------------------------------------------------------
print()
print("=" * 68)
print("  ATTACK SUMMARY")
print("=" * 68)
results = [
    ("Random text",                random_result["score"]),
    ("Naive spoof (pos-agnostic)", naive_result["score"]),
    ("Position-aware spoof",       aware_result["score"]),
    ("Top-3 candidate spoof",      top3_result["score"]),
    ("Genuine watermarked",        genuine_result["score"]),
]
for label, score in results:
    flag = "  <-- above threshold" if score > 0.55 else ""
    print(f"  {label:<30} {score:.4f}{flag}")
print()
print(f"  Detection threshold : 0.55")
print(f"  Oracle queries used : {N_QUERIES}")
print(f"  Green List size     : {len(green_list)} tokens")
print("=" * 68)

# Store for plotting
scores_for_plot = {label: score for label, score in results}
texts_collected = {
    "random" : random_text,
    "naive"  : naive_text,
    "aware"  : aware_text,
    "top3"   : top3_text,
    "genuine": genuine_text,
}


---
## Part 5 — Results Visualisation


In [ ]:
# Cell 7 — Results visualisation
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

DARK      = "#0D1B2A"
NAVY      = "#1A2B4A"
SLATE     = "#334155"
LIGHT     = "#CBD5E1"
WHITE     = "#FFFFFF"
CYAN      = "#00B4D8"
GREEN     = "#4CAF50"
ORANGE    = "#FF7043"
GOLD      = "#FFD700"
GRAY      = "#64748B"

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor(DARK)

# --- Left: detector scores bar chart -------------------------------------
ax1.set_facecolor(NAVY)
labels = list(scores_for_plot.keys())
values = list(scores_for_plot.values())
colors = [GRAY, SLATE, CYAN, GREEN, GOLD]

bars = ax1.bar(labels, values, color=colors, width=0.52, zorder=3)
ax1.axhline(0.55, color=ORANGE, linewidth=2, linestyle="--",
            zorder=4, label="Detection threshold (0.55)")
ax1.axhline(0.50, color=WHITE,  linewidth=1, linestyle=":",
            zorder=4, alpha=0.35, label="Random baseline (0.50)")
ax1.set_ylim(0.35, 1.05)
ax1.set_ylabel("Mean G-Value Score", color=WHITE, fontsize=11)
ax1.set_title("Detector Scores by Text Type", color=WHITE, fontsize=12, pad=10)
ax1.tick_params(colors=WHITE, labelsize=8)
ax1.set_xticklabels(labels, color=WHITE, rotation=14, ha="right")
for spine in ax1.spines.values():
    spine.set_edgecolor(SLATE)
for bar, val in zip(bars, values):
    ax1.text(bar.get_x() + bar.get_width() / 2, val + 0.012,
             f"{val:.4f}", ha="center", va="bottom",
             color=WHITE, fontweight="bold", fontsize=9)
ax1.yaxis.grid(True, color=SLATE, linewidth=0.7, zorder=0)
ax1.legend(facecolor=DARK, labelcolor=WHITE, fontsize=8)

# --- Right: Z-score distribution -----------------------------------------
ax2.set_facecolor(NAVY)
z_arr    = np.array(all_z_scores)
z_normal = z_arr[z_arr <= Z_THRESHOLD]
z_green  = z_arr[z_arr >  Z_THRESHOLD]

ax2.hist(z_normal, bins=40, color=SLATE, alpha=0.9,
         label=f"Normal tokens  ({len(z_normal)})", zorder=3)
ax2.hist(z_green,  bins=15, color=CYAN,  alpha=0.9,
         label=f"Green List     ({len(z_green)})",  zorder=4)
ax2.axvline(Z_THRESHOLD, color=ORANGE, linewidth=2, linestyle="--",
            label=f"Z threshold ({Z_THRESHOLD})", zorder=5)
ax2.set_xlabel("Z-Score", color=WHITE, fontsize=11)
ax2.set_ylabel("Token Count",  color=WHITE, fontsize=11)
ax2.set_title("Z-Score Distribution — Token Frequency Analysis",
              color=WHITE, fontsize=12, pad=10)
ax2.tick_params(colors=WHITE)
for spine in ax2.spines.values():
    spine.set_edgecolor(SLATE)
ax2.yaxis.grid(True, color=SLATE, linewidth=0.7, zorder=0)
ax2.legend(facecolor=DARK, labelcolor=WHITE, fontsize=8)

plt.tight_layout(pad=2.5)
plt.savefig("attack_results.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("Saved: attack_results.png")
